# Private CNN context expert inference

The CNN architecture, hand features, and S3 context layout credit [Briano (woominyo)](https://www.kaggle.com/code/woominyo/lb-0-890-wear-hasca-2026-timeline-graph). All models were trained locally. This notebook only loads fitted weights and runs inference for the method validated on held-out subjects.


In [ ]:
import os
for _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_k, "4")       # Kaggle has 4 vCPUs; the reference runs used 4 BLAS threads as well
import gc, hashlib, json, re, shutil, subprocess, sys, threading, time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import linear_sum_assignment
import lightgbm as lgb

SMOKE = os.environ.get("WEAR_SMOKE", "0") == "1"      # tiny run for testing the code
SMOKE_SUBJECTS = {0, 1, 2, 3, 4}
SLUG = "3rd-wear-dataset-challenge-hasca-2026"
ON_KAGGLE = Path("/kaggle/input").exists()


def find_input():
    if os.environ.get("WEAR_DATA"):
        return Path(os.environ["WEAR_DATA"])
    for p in (Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG):
        if (p / "test" / "test_meta_data.csv").exists():
            return p
    hits = sorted(Path("/kaggle/input").glob("*/test/test_meta_data.csv")) + \
        sorted(Path("/kaggle/input").glob("*/*/test/test_meta_data.csv"))
    assert hits, "competition data not found: attach the competition or set WEAR_DATA"
    return hits[0].parent.parent


INPUT = find_input()
WORK = Path(os.environ.get("WEAR_WORK", "/kaggle/working" if ON_KAGGLE else "wear_run")).resolve()   # submission.csv
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
if os.environ.get("WEAR_ROOT"):
    ROOT = Path(os.environ["WEAR_ROOT"])
elif ON_KAGGLE and shutil.disk_usage("/tmp").free > 12e9:
    ROOT = Path("/tmp/wear")            # about 6.5 GB of scratch that must not become notebook output
else:
    ROOT = WORK / "_wear"
PROC, RUNS_DIR = ROOT / "proc", ROOT / "runs"
for _p in (ROOT, PROC, RUNS_DIR):
    _p.mkdir(parents=True, exist_ok=True)
CLEANUP = os.environ.get("WEAR_CLEANUP", "1" if ON_KAGGLE else "0") == "1"
WINDOW_BUDGET_MIN = float(os.environ.get("WEAR_WINDOW_BUDGET_MIN", "150"))   # wall-clock cap of the GPU stage

T_START, TIMES = time.time(), {}


def rss_gb():
    try:
        s = Path("/proc/self/status").read_text()
        return tuple(int(re.search(k + r":\s+(\d+)", s).group(1)) / 2 ** 20 for k in ("VmRSS", "VmHWM"))
    except Exception:
        return float("nan"), float("nan")


class stage:
    """logs wall time and memory of a pipeline stage and frees memory afterwards"""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f"\n===== {self.name} | start at {(self.t - T_START) / 60:.1f} min", flush=True)
        return self

    def __exit__(self, *exc):
        gc.collect()
        TIMES[self.name] = time.time() - self.t
        r, h = rss_gb()
        print(f"===== {self.name} | {TIMES[self.name] / 60:.1f} min (total {(time.time() - T_START) / 60:.1f} min) "
              f"| RSS {r:.1f} GB, peak {h:.1f} GB", flush=True)


def error_line(text):
    """the most informative line of a subprocess traceback: the last 'SomeError: message' line"""
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    hits = [l for l in lines if re.match(r"[A-Za-z_][\w.]*(Error|Exception)(:|$)", l)]
    return (hits or lines or ["?"])[-1]


# Is there a GPU that torch can actually use? This runs in a subprocess so that the notebook itself holds no CUDA context.
# A GPU can be visible but unusable: torch.cuda.is_available() is True, yet the torch build has no kernels for it
# (e.g. a P100 = sm_60 with a torch build for sm_75+). The first CUDA op then fails, so the check runs one.
_chk = subprocess.run([sys.executable, "-c",
                       "import json, torch; assert torch.cuda.is_available(), 'torch.cuda.is_available() is False'; "
                       "torch.ones(8, device='cuda').sum().item(); "
                       "print(json.dumps({'n': torch.cuda.device_count(), 'name': torch.cuda.get_device_name(0), "
                       "'cap': torch.cuda.get_device_capability(0), 'torch': torch.__version__}))"],
                      capture_output=True, text=True)
GPU_OK = _chk.returncode == 0
GPU_INFO = json.loads(_chk.stdout.strip().splitlines()[-1]) if GPU_OK else {"n": 0}
N_GPU = GPU_INFO["n"]
print(f"input {INPUT}\nwork {WORK}\nscratch {ROOT}\nSMOKE={SMOKE} GPU={GPU_INFO if GPU_OK else 'none'}")
if not GPU_OK:
    print("WARNING: no usable GPU (" + error_line(_chk.stderr) + ").\n"
          "The window models fall back to a very reduced CPU run (with CUDA hidden from the training subprocesses), "
          "so the score will be far below 0.903. Select the GPU T4 x2 accelerator to reproduce the LB score.")
_model_hits = list(Path('/kaggle/input').rglob('link_model.txt'))
assert len(_model_hits) == 1, f'Expected one attached model dataset: {_model_hits}'
MODEL_ROOT = _model_hits[0].parent
assert GPU_OK and N_GPU >= 2, 'This public kernel requires T4 x2'
print('model_root', MODEL_ROOT, flush=True)

_context_hits = list(Path('/kaggle/input').rglob('s3_model.txt'))
assert len(_context_hits) == 1, f'Expected one attached private context dataset: {_context_hits}'
CONTEXT_ROOT = _context_hits[0].parent
print('context_model_root', CONTEXT_ROOT, flush=True)


In [ ]:
CLASSES = ["null", "jogging", "jogging (rotating arms)", "jogging (skipping)", "jogging (sidesteps)",
           "jogging (butt-kicks)", "stretching (triceps)", "stretching (lunging)", "stretching (shoulders)",
           "stretching (hamstrings)", "stretching (lumbar rotation)", "push-ups", "push-ups (complex)",
           "sit-ups", "sit-ups (complex)", "burpees", "lunges", "lunges (complex)", "bench-dips"]
LABEL_IDX = {c: i for i, c in enumerate(CLASSES)}
SENSORS = ["right_arm", "right_leg", "left_leg", "left_arm"]
ACC_COLS = [f"{s}_acc_{a}" for s in SENSORS for a in "xyz"]


with stage("W1 test preprocessing"):
    meta = pd.read_csv(INPUT / "test" / "test_meta_data.csv")
    t_acc = np.load(INPUT / "test" / "test_inertial_data.npy").astype(np.float32)
    t_vid = np.load(INPUT / "test" / "test_videomae_data.npy")
    print("test shapes:", t_acc.shape, t_vid.shape, t_vid.dtype)
    if t_vid.shape[1] == 768:
        t_vid = t_vid.transpose(0, 2, 1)
    loc_col = "sensor_location" if "sensor_location" in meta else "inertial_sensor_location"
    np.save(PROC / "test_acc.npy", t_acc)
    np.save(PROC / "test_vid.npy", np.ascontiguousarray(t_vid).astype(np.float16))
    np.save(PROC / "test_sensor.npy", meta[loc_col].map({s: i for i, s in enumerate(SENSORS)}).to_numpy(np.int8))
    np.save(PROC / "test_id.npy", meta["id"].to_numpy())
    np.save(PROC / "test_sbj.npy", meta["sbj_id"].to_numpy(np.int16))
    assert len(meta) == len(t_acc) == len(t_vid)
    print(meta[loc_col].value_counts().to_dict(), meta["sbj_id"].value_counts().sort_index().to_dict())


In [ ]:
%%writefile window_infer.py
# Derived from Briano (woominyo), WEAR Timeline + Graph, version 7.
# https://www.kaggle.com/code/woominyo/lb-0-890-wear-hasca-2026-timeline-graph
# Model architecture, normalization and numerical operations are retained.
import os
import time
T_INFER_START = time.perf_counter()
"""Load source window classifiers and infer the test windows on one GPU."""
import argparse
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

p = argparse.ArgumentParser()
p.add_argument("--data", default="/workspace/proc")
p.add_argument("--out", default="/workspace/runs/v1")
p.add_argument("--dim", type=int, default=256)
p.add_argument("--vnorm", default="global", choices=["global", "subject"])  # video feature normalisation unit
p.add_argument("--anorm", default="global", choices=["global", "subject"])  # IMU normalisation unit
p.add_argument("--vid_arch", default="pool", choices=["pool", "tf"])  # pool: mean/std pooling + MLP
p.add_argument("--vdrop", type=float, default=0.5)  # video input feature dropout
p.add_argument("--only", default="none", choices=["none", "imu", "video"])  # single-modality model
args = p.parse_args()

N_CLS, WIN, VLEN, STRIDE = 19, 50, 15, 5
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# PATCH (a): bf16 autocast is unsupported or emulated on T4 / P100 -> capability switch
CAP = torch.cuda.get_device_capability() if dev.type == "cuda" else (0, 0)
USE_AMP = CAP >= (7, 0)                                             # T4 (7,5): fp16 tensor cores; P100 (6,0): fp32
USE_AMP = USE_AMP and os.environ.get("WEAR_AMP", "1") != "0"          # the driver retries a failed run in fp32
AMP_DTYPE = torch.float16 if (7, 0) <= CAP < (8, 0) else torch.bfloat16
torch.ones(8, device=dev).sum().item()                              # fails fast if torch has no kernels for this GPU
if dev.type == "cuda":
    torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
out_dir = Path(args.out)
out_dir.mkdir(parents=True, exist_ok=True)
log_f = open(out_dir / "log.txt", "a")


def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    log_f.write(s + "\n")
    log_f.flush()


log(f"device {dev} capability {CAP} precision {AMP_DTYPE if USE_AMP else torch.float32} torch {torch.__version__}")


# ---------------- data ----------------
D = Path(args.data)
assert args.anorm == args.vnorm == "subject"

test_acc_np = np.load(D / "test_acc.npy")
test_vid_np = np.load(D / "test_vid.npy")
test_sensor_np = np.load(D / "test_sensor.npy").astype(np.int64)
test_sbj_np = np.load(D / "test_sbj.npy").astype(np.int64)

# normalisation groups: train subjects + test subjects (global mode: all groups share one set of stats)
SB = list(range(22))  # source training has 22 subjects; preserves test normalization group offsets
TS = sorted(set(test_sbj_np.tolist()))
G = len(SB) + len(TS)
test_grp_np = np.array([len(SB) + TS.index(t) for t in test_sbj_np])


def acc_stats(x, ok):
    """x (n,4,3), ok (n,4) -> per-sensor mean/std (4,3)"""
    m, sd = np.zeros((4, 3)), np.ones((4, 3))
    for k in range(4):
        xs = x[ok[:, k], k].astype(np.float64)
        if len(xs) > 100:
            m[k], sd[k] = xs.mean(0), xs.std(0) + 1e-3
    return m, sd


def vid_stats(v):
    v = v.astype(np.float32)
    return v.mean(0), v.std(0) + 1e-3


AM, AS = np.zeros((G, 4, 3)), np.ones((G, 4, 3))
VM, VS = np.zeros((G, 768)), np.ones((G, 768))
for ti, t in enumerate(TS):
    m = test_sbj_np == t
    x = np.zeros((m.sum(), WIN, 4, 3), np.float32)
    ok = np.zeros((m.sum(), WIN, 4), bool)
    x[np.arange(m.sum()), :, test_sensor_np[m]] = test_acc_np[m]
    ok[np.arange(m.sum()), :, test_sensor_np[m]] = True
    AM[len(SB) + ti], AS[len(SB) + ti] = acc_stats(x.reshape(-1, 4, 3), ok.reshape(-1, 4))
    VM[len(SB) + ti], VS[len(SB) + ti] = vid_stats(test_vid_np[m].reshape(-1, 768))
AM, AS, VM, VS = (torch.tensor(x, dtype=torch.float32, device=dev) for x in (AM, AS, VM, VS))

def prep_inputs(a_raw, v_raw, sensor, g):
    """a_raw (B,50,3) in g, v_raw (B,15,768) f16, sensor (B,), g (B,) normalisation group -> model inputs"""
    a_raw = a_raw.float()
    mag = a_raw.norm(dim=-1, keepdim=True) - 1.0
    a = (a_raw - AM[g, sensor][:, None]) / AS[g, sensor][:, None]
    a = torch.cat([a, mag], -1)
    v = (v_raw.float() - VM[g][:, None]) / VS[g][:, None]
    if args.only != "none":
        if args.only == "imu":
            v = torch.zeros_like(v)
        else:
            a = torch.zeros_like(a)
    return a, v


# ---------------- model ----------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, k, d=1):
        super().__init__()
        self.c = nn.Conv1d(cin, cout, k, padding=d * (k // 2), dilation=d)
        self.bn = nn.BatchNorm1d(cout)
        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        return F.gelu(self.bn(self.c(x))) + self.skip(x)


class Net(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.sens = nn.Embedding(4, 32)
        self.imu = nn.Sequential(ConvBlock(4 + 32, 64, 7), ConvBlock(64, 128, 5), ConvBlock(128, 128, 5, 2),
                                 ConvBlock(128, d, 3, 4), ConvBlock(d, d, 3, 8))
        self.imu_out = nn.Sequential(nn.Linear(2 * d, d), nn.GELU())
        self.vin = nn.Sequential(nn.Dropout(args.vdrop), nn.Linear(768, d), nn.GELU(), nn.Dropout(0.2))
        if args.vid_arch == "tf":
            self.vpos = nn.Parameter(torch.zeros(1, VLEN, d))
            layer = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, activation="gelu", batch_first=True, norm_first=True)
            self.vtf = nn.TransformerEncoder(layer, 2)
        self.vid_out = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.3))
        self.fuse = nn.Sequential(nn.Linear(2 * d + 32, d), nn.GELU(), nn.Dropout(0.3), nn.Linear(d, N_CLS))
        self.h_imu = nn.Linear(d, N_CLS)
        self.h_vid = nn.Linear(d, N_CLS)

    def forward(self, a, v, sensor):
        e = self.sens(sensor)
        x = torch.cat([a.transpose(1, 2), e[:, :, None].expand(-1, -1, a.shape[1])], 1)
        h = self.imu(x)
        hi = self.imu_out(torch.cat([h.mean(-1), h.amax(-1)], 1))
        if args.vid_arch == "tf":
            z = self.vtf(self.vin(v) + self.vpos)
            hv = self.vid_out(torch.cat([z.mean(1), z.amax(1)], 1))
        else:
            z = self.vin(v)
            hv = self.vid_out(torch.cat([z.mean(1), z.std(1)], 1))
        return self.fuse(torch.cat([hi, hv, e], 1)), self.h_imu(hi), self.h_vid(hv)


@torch.no_grad()
def predict(model, a_raw, v_raw, sensor, g, bs=2048, heads=False):
    model.eval()
    outs = []
    for i in range(0, len(sensor), bs):
        a, v = prep_inputs(a_raw[i:i + bs], v_raw[i:i + bs], sensor[i:i + bs], g[i:i + bs])
        with torch.autocast(dev.type, dtype=AMP_DTYPE, enabled=USE_AMP):  # PATCH (a)
            logits = model(a, v, sensor[i:i + bs])
        outs.append(torch.stack([F.log_softmax(x.float(), 1) for x in logits]))
    out = torch.cat(outs, 1).cpu().numpy()  # (3, N, C): fusion, imu-only, video-only
    return out if heads else out[0]


@torch.no_grad()
def vid_emb(v_raw, g, bs=4096):
    """for neighbour search: window mean of the normalised video features"""
    return torch.cat([((v_raw[i:i + bs].float() - VM[g[i:i + bs]][:, None]) / VS[g[i:i + bs]][:, None]).mean(1)
                      for i in range(0, len(g), bs)])


# Inference-only window-model runner. One process owns one GPU and handles three
# source-model runs so the normalization arrays and input tensors are loaded once.
from pathlib import Path
MODEL_ROOT = Path(os.environ["WEAR_MODEL_ROOT"])
RUN_NAMES = os.environ["WEAR_RUN_NAMES"].split(",")
BATCH = int(os.environ.get("WEAR_INFER_BATCH", "2048"))
device_name = torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu"
print(f"setup_s={time.perf_counter()-T_INFER_START:.3f} device={device_name} runs={RUN_NAMES}", flush=True)
test_acc = torch.from_numpy(test_acc_np).to(dev)
test_vid = torch.from_numpy(test_vid_np).to(dev)
test_sensor = torch.from_numpy(test_sensor_np).to(dev)
test_grp = torch.from_numpy(test_grp_np).to(dev)

# Prepare once; the original called prep_inputs once per model and batch.
parts_a, parts_v = [], []
args.only = "none"
for lo in range(0, len(test_sensor), BATCH):
    aa, vv = prep_inputs(test_acc[lo:lo+BATCH], test_vid[lo:lo+BATCH],
                         test_sensor[lo:lo+BATCH], test_grp[lo:lo+BATCH])
    parts_a.append(aa)
    parts_v.append(vv)
prepared_a, prepared_v = torch.cat(parts_a), torch.cat(parts_v)
del parts_a, parts_v
print(f"preprocess_s={time.perf_counter()-T_INFER_START:.3f} rows={len(test_sensor)}", flush=True)

@torch.no_grad()
def predict_prepared(model, only):
    model.eval()
    chunks = []
    batch_times = []
    for lo in range(0, len(test_sensor), BATCH):
        a = prepared_a[lo:lo+BATCH]
        v = prepared_v[lo:lo+BATCH]
        if only == "imu":
            v = torch.zeros_like(v)
        if dev.type == "cuda":
            torch.cuda.synchronize()
        batch_start = time.perf_counter()
        with torch.autocast(dev.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(a, v, test_sensor[lo:lo+BATCH])
        chunks.append(F.log_softmax(logits[0].float(), 1))
        if dev.type == "cuda":
            torch.cuda.synchronize()
        batch_times.append(time.perf_counter() - batch_start)
    return torch.cat(chunks).cpu().numpy(), batch_times

for run_name in RUN_NAMES:
    args.only = "imu" if run_name.startswith("imu") else "none"
    outputs = []
    start = time.perf_counter()
    for fold in range(5):
        model = Net(args.dim).to(dev)
        state = torch.load(MODEL_ROOT / f"{run_name}_model_f{fold}.pt", map_location=dev, weights_only=True)
        model.load_state_dict(state)
        candidate, batch_times = predict_prepared(model, args.only)
        if fold == 0:
            # One full-size batch checks the optimized path against the source.
            reference = predict(model, test_acc[:BATCH], test_vid[:BATCH],
                                test_sensor[:BATCH], test_grp[:BATCH], bs=BATCH)
            delta = float(np.max(np.abs(reference - candidate[:BATCH])))
            print(f"{run_name} source_vs_prepared_max_abs={delta:.8g}", flush=True)
            assert delta < 1e-5
            next_lo = BATCH if len(test_sensor) > BATCH else 0
            source_start = time.perf_counter()
            reference_next = predict(model, test_acc[next_lo:next_lo+BATCH],
                                     test_vid[next_lo:next_lo+BATCH],
                                     test_sensor[next_lo:next_lo+BATCH],
                                     test_grp[next_lo:next_lo+BATCH], bs=BATCH)
            source_batch_s = time.perf_counter() - source_start
            next_delta = float(np.max(np.abs(reference_next - candidate[next_lo:next_lo+BATCH])))
            assert next_delta < 1e-5
            print(f"{run_name} source_batch_s={source_batch_s:.4f} "
                  f"prepared_batch_s={np.median(batch_times[1:] or batch_times):.4f}", flush=True)
        outputs.append(candidate)
        del model, state
    np.save(out_dir / f"{run_name}_test_logp_raw.npy", np.mean(outputs, 0))
    del outputs
    print(f"{run_name} inference_s={time.perf_counter()-start:.3f}", flush=True)

if "final_s0" in RUN_NAMES:
    np.save(out_dir / "test_emb.npy", vid_emb(test_vid, test_grp).half().cpu().numpy())
print(f"total_s={time.perf_counter()-T_INFER_START:.3f}", flush=True)


In [ ]:
with stage("W3 pretrained GPU window inference"):
    model_names = sorted(p.name.rsplit("_model_f0.pt", 1)[0]
                         for p in MODEL_ROOT.glob("*_model_f0.pt"))
    assert model_names and any(n.startswith("final") for n in model_names)
    assert any(n.startswith("imu") for n in model_names)
    source_precision = (MODEL_ROOT / "source_precision.txt").read_text().strip()
    assert source_precision in ("cpu_fp32", "gpu_amp")
    jobs = ((0, tuple(model_names[::2])), (1, tuple(model_names[1::2])))
    processes = []
    for gpu, names in jobs:
        output = RUNS_DIR / f"gpu_{gpu}"
        output.mkdir(exist_ok=True)
        env = dict(os.environ)
        env.update(CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="1",
                   WEAR_MODEL_ROOT=str(MODEL_ROOT), WEAR_RUN_NAMES=",".join(names), WEAR_INFER_BATCH="2048",
                   WEAR_AMP="0" if source_precision == "cpu_fp32" else "1")
        command = [sys.executable, str(WORK / "window_infer.py"), "--data", str(PROC),
                   "--out", str(output), "--vnorm", "subject", "--anorm", "subject", "--vid_arch", "pool"]
        log_file = (output / "inference.log").open("w")
        process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, env=env)
        processes.append((gpu, names, process, log_file, output))
    for gpu, names, process, log_file, output in processes:
        code = process.wait()
        log_file.close()
        log = (output / "inference.log").read_text()
        print(log, flush=True)
        assert code == 0, f"GPU {gpu} inference failed: {log[-3000:]}"
        for name in names:
            dest = RUNS_DIR / name
            dest.mkdir(exist_ok=True)
            shutil.copy2(output / f"{name}_test_logp_raw.npy", dest / "test_logp_raw.npy")
        if (output / "test_emb.npy").exists():
            shutil.copy2(output / "test_emb.npy", RUNS_DIR / "final_s0" / "test_emb.npy")
    FUSION = [RUNS_DIR / name for name in model_names if name.startswith("final")]
    IMU = [RUNS_DIR / name for name in model_names if name.startswith("imu")]


In [ ]:
with stage("W4 fixed-CV window blend"):
    model_names = sorted(p.name.rsplit("_model_f0.pt", 1)[0]
                         for p in MODEL_ROOT.glob("*_model_f0.pt"))
    def normalized_run(name):
        x = np.load(RUNS_DIR / name / "test_logp_raw.npy")
        m = x.max(1, keepdims=True)
        return x - m - np.log(np.exp(x - m).sum(1, keepdims=True))
    fus = np.mean([normalized_run(name) for name in model_names
                   if name.startswith("final")], axis=0)
    imu = np.mean([normalized_run(name) for name in model_names
                   if name.startswith("imu")], axis=0)
    logits = fus + .2 * imu
    logits -= logits.max(1, keepdims=True)
    prob = np.exp(logits)
    prob /= prob.sum(1, keepdims=True)
    emb = np.load(RUNS_DIR / "final_s0" / "test_emb.npy").astype(np.float32)
    subjects = np.load(PROC / "test_sbj.npy")
    sensor = np.load(PROC / "test_sensor.npy")


In [ ]:
with stage("L1 mutual video-boundary links"):
    import torch
    from scipy import sparse
    video = np.load(PROC / "test_vid.npy", mmap_mode="r")
    src, dst, link_values = [], [], []
    for subject in np.unique(subjects):
        ii = np.flatnonzero(subjects == subject)
        v = torch.as_tensor(np.asarray(video[ii], np.float32), device="cuda")
        v = torch.nn.functional.normalize(v, dim=2)
        head = torch.nn.functional.normalize(v[:, :3].mean(1), dim=1)
        tail = torch.nn.functional.normalize(v[:, -3:].mean(1), dim=1)
        sim = tail @ head.T + .5 * (v[:, -1] @ v[:, 0].T)
        sim.fill_diagonal_(-torch.inf)
        successor_local = sim.argmax(1)
        predecessor_local = sim.argmax(0)
        rows = torch.arange(len(ii), device="cuda")
        picked = rows[predecessor_local[successor_local] == rows]
        src.append(ii[picked.cpu().numpy()])
        dst.append(ii[successor_local[picked].cpu().numpy()])
        link_values.append(sim[picked, successor_local[picked]].cpu().numpy())
    src, dst, link_values = np.concatenate(src), np.concatenate(dst), np.concatenate(link_values)
    successor = np.full(len(subjects), -1, np.int64)
    link_score = np.full(len(subjects), -50, np.float32)
    successor[src] = dst
    link_score[src] = link_values
    print('mutual_links', len(src), 'rows', len(subjects))


In [ ]:
BANDS = [(1, 1), (2, 2), (3, 4), (5, 7), (8, 12), (13, 25)]   # Hz (50 samples -> 1 Hz resolution)


def imu_feats(acc):
    """acc (N,50,3) in g -> (N,130) features, names"""
    acc = acc.astype(np.float32)
    mag = np.linalg.norm(acc, axis=2, keepdims=True)
    X = np.concatenate([acc, mag], 2)  # (N,50,4)
    chn = ["x", "y", "z", "m"]
    feats, names = [], []

    def add(v, nm):
        feats.append(v.astype(np.float32))
        names.append(nm)

    mu = X.mean(1)
    Xd = X - mu[:, None]
    sd = X.std(1) + 1e-6
    q = np.percentile(X, [10, 25, 50, 75, 90], axis=1)  # (5,N,4)
    dif = np.diff(X, axis=1)
    F = np.abs(np.fft.rfft(Xd, axis=1)) ** 2  # (N,26,4), 0..25 Hz
    Ftot = F[:, 1:].sum(1) + 1e-9
    pn = F[:, 1:] / Ftot[:, None]
    ac_full = np.stack([(Xd[:, :-L] * Xd[:, L:]).mean(1) for L in range(3, 41)], 1) / (sd[:, None] ** 2)  # (N,38,4)
    for c in range(4):
        n = chn[c]
        add(mu[:, c], f"mean_{n}")
        add(sd[:, c], f"std_{n}")
        add(X[:, :, c].min(1), f"min_{n}")
        add(X[:, :, c].max(1), f"max_{n}")
        for k, p_ in enumerate([10, 25, 50, 75, 90]):
            add(q[k, :, c], f"p{p_}_{n}")
        add(q[3, :, c] - q[1, :, c], f"iqr_{n}")
        add(X[:, :, c].max(1) - X[:, :, c].min(1), f"range_{n}")
        add((Xd[:, :, c] ** 2).mean(1), f"energy_{n}")
        add(np.abs(Xd[:, :, c]).mean(1), f"mad_{n}")
        z = Xd[:, :, c] / sd[:, c:c + 1]
        add((z ** 3).mean(1), f"skew_{n}")
        add((z ** 4).mean(1), f"kurt_{n}")
        s = np.sign(Xd[:, :, c])
        add((s[:, 1:] * s[:, :-1] < 0).mean(1), f"zcr_{n}")
        add(np.abs(dif[:, :, c]).mean(1), f"jerk_{n}")
        add(dif[:, :, c].std(1), f"jerkstd_{n}")
        add(np.argmax(F[:, 1:, c], 1) + 1.0, f"domf_{n}")
        add(F[:, 1:, c].max(1) / Ftot[:, c], f"domfrac_{n}")
        add(np.log(Ftot[:, c]), f"logpow_{n}")
        for lo, hi in BANDS:
            add(F[:, lo:hi + 1, c].sum(1) / Ftot[:, c], f"band{lo}-{hi}_{n}")
        add(-(pn[:, :, c] * np.log(pn[:, :, c] + 1e-12)).sum(1), f"spent_{n}")
        add(ac_full[:, :, c].max(1), f"acmax_{n}")
        add(np.argmax(ac_full[:, :, c], 1) + 3.0, f"aclag_{n}")
        add(X[:, -10:, c].mean(1) - X[:, :10, c].mean(1), f"trend_{n}")
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        add((Xd[:, :, a] * Xd[:, :, b]).mean(1) / (sd[:, a] * sd[:, b]), f"corr_{chn[a]}{chn[b]}")
    g = mu[:, :3] / (np.linalg.norm(mu[:, :3], axis=1, keepdims=True) + 1e-6)
    for k in range(3):
        add(g[:, k], f"grav_{chn[k]}")
    return np.stack(feats, 1), names


def standardize(X, sbj, sensor):
    """z-score within (subject x limb), using only that subject's own windows (label-free)"""
    Z = np.empty_like(X)
    for s in np.unique(sbj):
        for k in np.unique(sensor):
            ii = np.flatnonzero((sbj == s) & (sensor == k))
            if not len(ii):
                continue
            m, sd = X[ii].mean(0), X[ii].std(0) + 1e-6
            Z[ii] = (X[ii] - m) / sd
    return Z


def video_pass(V, n, chunk=3000):
    """frame mean (N,768) + motion scalars (N,4)"""
    vm = np.zeros((n, 768), np.float32)
    mot = np.zeros((n, 4), np.float32)
    for a in range(0, n, chunk):
        v = np.asarray(V[a:a + chunk], np.float32)
        vm[a:a + chunk] = v.mean(1)
        u = v / (np.linalg.norm(v, axis=2, keepdims=True) + 1e-6)
        dd = np.linalg.norm(np.diff(v, axis=1), axis=2)
        mot[a:a + chunk, 0] = dd.mean(1)
        mot[a:a + chunk, 1] = (u[:, 0] * u[:, -1]).sum(1)
        mot[a:a + chunk, 2] = (u[:, :-1] * u[:, 1:]).sum(2).min(1)
        mot[a:a + chunk, 3] = v.std(1).mean(1)
    return vm, mot


def hand_features(dd):
    imu, names = imu_feats(dd["acc"]())
    z = standardize(imu, dd["sbj"], dd["sensor"])
    X = np.concatenate([imu, z, dd["sensor"][:, None].astype(np.float32)], 1).astype(np.float32)
    vm, mot = video_pass(dd["vid"](), len(dd["sbj"]))
    return {"imu": X, "vmean": vm, "vmot": mot}, names + [f"z_{n}" for n in names] + ["sensor"]


K_PCA = 48


def sub_center(vm, sbj):
    mu = np.zeros((sbj.max() + 1, vm.shape[1]), np.float32)
    for s in np.unique(sbj):
        mu[s] = vm[sbj == s].mean(0)
    return vm - mu[sbj], mu


def pca(X):
    m = X.mean(0)
    C = np.cov((X - m).T)
    ev, evec = np.linalg.eigh(C)
    o = np.argsort(-ev)[:K_PCA]
    return m.astype(np.float32), evec[:, o].astype(np.float32), ev[o]


def tstd(V, comps, chunk=3000):
    """per-frame projection -> std over the 15 frames"""
    n = V.shape[0]
    out = np.zeros((n, comps.shape[1]), np.float32)
    for a in range(0, n, chunk):
        v = np.asarray(V[a:a + chunk], np.float32)
        out[a:a + chunk] = (v @ comps).std(1)
    return out



NB_IMU = ('mean_x', 'mean_y', 'mean_z', 'std_m', 'energy_m', 'mad_x', 'mad_y', 'mad_z', 'jerk_m', 'domf_m', 'acmax_m', 'aclag_m', 'logpow_m', 'grav_x', 'grav_y', 'grav_z')
def neighbor_block(logp: np.ndarray, imu: np.ndarray, sensor: np.ndarray,
                   successor: np.ndarray, score: np.ndarray,
                   names: list[str]) -> np.ndarray:
    n = len(logp)
    previous = np.full(n, -1, np.int64)
    valid = successor >= 0
    previous[successor[valid]] = np.flatnonzero(valid)

    def take(x: np.ndarray, index: np.ndarray) -> np.ndarray:
        out = np.asarray(x[np.maximum(index, 0)], np.float32).copy()
        out[index < 0] = np.nan
        return out

    next2 = np.where(valid, successor[np.maximum(successor, 0)], -1)
    prev2 = np.where(previous >= 0, previous[np.maximum(previous, 0)], -1)
    p = np.exp(logp)
    acc = np.zeros_like(p)
    count = np.zeros(n, np.float32)
    for idx in (prev2, previous, successor, next2):
        acc += np.where(idx[:, None] >= 0, p[np.maximum(idx, 0)], 0)
        count += idx >= 0
    context = np.log(np.where(count[:, None] > 0,
                              acc / np.maximum(count[:, None], 1), np.nan) + 1e-6)
    zi = [names.index("z_" + name) for name in NB_IMU]
    sensor = sensor[:, None].astype(np.float32)
    parts = [take(logp, successor), take(logp, previous), context,
             np.where(valid, score, np.nan)[:, None],
             np.where(previous >= 0, score[np.maximum(previous, 0)], np.nan)[:, None],
             count[:, None], take(sensor, successor), take(sensor, previous),
             take(imu[:, zi], successor), take(imu[:, zi], previous)]
    result = np.concatenate(parts, 1).astype(np.float32)
    assert result.shape == (n, 94), result.shape
    return result


In [ ]:
with stage("S3 context expert"):
    acc = np.load(PROC / "test_acc.npy")
    data = {"acc": lambda: acc, "vid": lambda: video,
            "sbj": subjects, "sensor": sensor}
    feat, names = hand_features(data)
    centered, _ = sub_center(feat.pop("vmean"), subjects)
    projection = np.load(CONTEXT_ROOT / "video_pca.npz")
    mean, components = projection["mean"], projection["components"]
    vpca = np.concatenate([(centered - mean) @ components,
                           tstd(video, components)], axis=1).astype(np.float32)
    x_t = np.concatenate([feat["imu"], feat["vmot"], vpca], axis=1).astype(np.float32)
    assert x_t.shape == (len(subjects), 361), x_t.shape
    saved_names = json.loads((CONTEXT_ROOT / "feature_names.json").read_text())
    assert names == saved_names, 'Feature order differs from local fit'
    lp = np.log(np.clip(prob, 1e-9, 1))
    nb = neighbor_block(lp, x_t[:, :261], sensor, successor, link_score, names)
    nn = np.empty((len(subjects), 21), np.float32)
    for subject in np.unique(subjects):
        ii = np.flatnonzero(subjects == subject)
        e = torch.as_tensor(emb[ii], dtype=torch.float32, device="cuda")
        e = torch.nn.functional.normalize(e, dim=1)
        sim = e @ e.T
        sim.fill_diagonal_(-torch.inf)
        value, neighbor = sim.topk(10, dim=1)
        neighbor = neighbor.cpu().numpy()
        value = value.cpu().numpy()
        nn[ii, :19] = np.log(prob[ii][neighbor].mean(1) + 1e-6)
        nn[ii, 19] = value.max(1)
        nn[ii, 20] = value.mean(1)
    x_s3 = np.concatenate([x_t, lp, fus, imu, nb, nn], axis=1).astype(np.float32)
    assert x_s3.shape == (len(subjects), 533), x_s3.shape
    expert = lgb.Booster(model_file=str(CONTEXT_ROOT / "s3_model.txt"))
    prob = expert.predict(x_s3).astype(np.float32)
    print('S3 raw classes', np.bincount(prob.argmax(1), minlength=19).tolist())
    del feat, centered, vpca, x_t, nb, nn, x_s3, expert, acc


In [ ]:
with stage("G1 subject video graph"):
    import torch
    from scipy import sparse
    graph = np.empty_like(prob)
    for subject in np.unique(subjects):
        ii = np.flatnonzero(subjects == subject)
        e = torch.as_tensor(emb[ii], dtype=torch.float32, device="cuda")
        e = torch.nn.functional.normalize(e, dim=1)
        q = torch.as_tensor(np.sqrt(prob[ii]), dtype=torch.float32, device="cuda")
        sim = e @ e.T + .5 * (q @ q.T)
        sim.fill_diagonal_(-torch.inf)
        k = min(10, len(ii) - 1)
        val, nb = sim.topk(k, dim=1)
        val, nb = val.cpu().numpy(), nb.cpu().numpy()
        rows = np.repeat(np.arange(len(ii)), k)
        w = sparse.csr_matrix((np.exp((val.ravel() - val.max()) / .1),
                               (rows, nb.ravel())), shape=(len(ii), len(ii)))
        w = w.maximum(w.T)
        degree = np.asarray(w.sum(1)).ravel()
        degree[degree == 0] = 1
        norm = sparse.diags(1 / np.sqrt(degree))
        w = norm @ w @ norm
        cur = prob[ii].copy()
        for _ in range(20):
            cur = .9 * (w @ cur) + .1 * prob[ii]
        graph[ii] = cur
    del emb, prob


In [ ]:
with stage("L2 video-link smoothing"):
    w = sparse.csr_matrix((np.ones(2 * len(src)),
                           (np.r_[src, dst], np.r_[dst, src])),
                          shape=(len(graph), len(graph)))
    degree = np.asarray(w.sum(1)).ravel()
    cur = graph.astype(np.float64)
    for _ in range(10):
        neighbor = (w @ cur) / np.maximum(degree[:, None], 1)
        cur = np.where(degree[:, None] > 0, .25 * graph + .75 * neighbor, graph)


In [ ]:
with stage("C1 circuit-count calibration"):
    calibrated = np.empty_like(cur)
    for subject in np.unique(subjects):
        ii = np.flatnonzero(subjects == subject)
        n = len(ii)
        null = max(n - 18 * 97, .05 * n)
        target = np.full(19, (n - null) / 18)
        target[0] = null
        q = np.clip(cur[ii], 1e-12, None) ** 2
        q /= q.sum(1, keepdims=True)
        for _ in range(50):
            q *= (target / np.maximum(q.sum(0), 1e-9))[None]
            q /= q.sum(1, keepdims=True)
        calibrated[ii] = q
    prediction = calibrated.argmax(1)
    ids = np.load(PROC / "test_id.npy")
    submission = pd.DataFrame({"id": ids, "target_feature": prediction.astype(int)})
    submission.to_csv(WORK / "submission.csv", index=False)
    print("submission", len(submission), submission.target_feature.value_counts().to_dict())


In [ ]:
print("stage times (min):")
for k, v in TIMES.items():
    print(f"  {k:60s} {v / 60:6.1f}")
print(f"total {(time.time() - T_START) / 60:.1f} min;", "submission:", WORK / "submission.csv")
if CLEANUP and ROOT.exists():
    shutil.rmtree(ROOT, ignore_errors=True)       # scratch arrays must not become notebook output